# Inventory Optimization for Perishable Goods
## FreshRetailNet-50K: From Newsvendor Theory to Non-Parametric Policies

---

**Objective**: Learn how to translate demand forecasts into optimal ordering decisions for perishable products.  
**Dataset**: FreshRetailNet-50K (store-product level daily sales with stockout indicators)  
**Key Question**: Given an uncertain demand forecast, how many units should we order to maximize profit?

### Notebook Outline

| Section | Topic | Key Concept |
|---------|-------|-------------|
| 1 | Setup & Generate Forecasts | LightGBM ensemble with quantile regression |
| 2 | The Newsvendor Problem | Critical ratio, optimal order quantity |
| 3 | The Gaussian Assumption Problem | Why Normal distribution fails for retail demand |
| 4 | Normal-Based Inventory Policies | Classical approaches and safety stock variants |
| 5 | Non-Normal Distribution Policies | Empirical, Zero-Inflated Gamma, KDE |
| 6 | Full Policy Comparison | Pareto frontier, profit ranking |
| 7 | Sensitivity Analysis | How cost parameters shape decisions |
| 8 | ABC-XYZ Inventory Segmentation | Classify items by value and demand predictability |
| 9 | Before vs After Comparison | Static rules → ML → ML + Optimization |
| 10 | Key Takeaways | Summary of lessons learned |

---
## Section 1: Setup & Generate Forecasts

We begin by loading the FreshRetailNet-50K dataset, sampling 3000 store-products stratified by stockout rate,
recovering censored demand, engineering features, and training a LightGBM ensemble.

The forecasting pipeline produces:
- **Point forecasts**: ensemble of MAE, Huber, and Q50 models
- **Quantile forecasts**: Q10, Q50, Q90 for uncertainty estimation
- **Historical standard deviation**: per store-product, used later for Normal-based policies

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, gamma
from sklearn.cluster import KMeans
import warnings
import gc

%matplotlib inline

import os
NB_OUT = os.path.join('..', 'notebook_output')
os.makedirs(NB_OUT, exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
warnings.filterwarnings('ignore')
np.random.seed(42)

print("Libraries loaded.")

### 1.1 Load Data & Stratified Sampling

In [ ]:
TRAIN_PATH = '../../data/freshretailnet/raw/data/train.parquet'
EVAL_PATH  = '../../data/freshretailnet/raw/data/eval.parquet'
N_SP = 3000

cols = [
    'city_id', 'store_id', 'management_group_id',
    'first_category_id', 'second_category_id', 'third_category_id',
    'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt',
    'discount', 'holiday_flag', 'activity_flag',
    'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'
]

train_raw = pd.read_parquet(TRAIN_PATH, columns=cols)
train_raw['sp'] = train_raw['store_id'] * 10000 + train_raw['product_id']

# Stratified sampling by stockout rate
so_rate = train_raw.groupby('sp')['stock_hour6_22_cnt'].apply(lambda x: (x > 0).mean())
so_rate = so_rate.reset_index()
so_rate.columns = ['sp', 'so_rate']
so_rate['bin'] = pd.qcut(so_rate['so_rate'], q=5, labels=False, duplicates='drop')

sampled = so_rate.groupby('bin').apply(
    lambda x: x.sample(min(len(x), N_SP // 5), random_state=42)
).reset_index(drop=True)['sp'].values

if len(sampled) < N_SP:
    extra = np.random.choice(
        list(set(so_rate['sp']) - set(sampled)),
        N_SP - len(sampled), replace=False
    )
    sampled = np.concatenate([sampled, extra])
sampled_set = set(sampled[:N_SP])

train = train_raw[train_raw['sp'].isin(sampled_set)].copy()
train['dt'] = pd.to_datetime(train['dt'])

ev_raw = pd.read_parquet(EVAL_PATH, columns=cols)
ev_raw['sp'] = ev_raw['store_id'] * 10000 + ev_raw['product_id']
ev = ev_raw[ev_raw['sp'].isin(sampled_set)].copy()
ev['dt'] = pd.to_datetime(ev['dt'])

# Downcast for memory efficiency
for c in train.select_dtypes('int64').columns:
    if c != 'sp':
        train[c] = train[c].astype('int32')
        ev[c] = ev[c].astype('int32')
for c in train.select_dtypes('float64').columns:
    train[c] = train[c].astype('float32')
    ev[c] = ev[c].astype('float32')

print(f"Train: {train.shape}  |  {train['dt'].min().date()} to {train['dt'].max().date()}")
print(f"Eval:  {ev.shape}  |  {ev['dt'].min().date()} to {ev['dt'].max().date()}")
print(f"Sampled SPs: {len(sampled_set):,}")
print(f"Stockout rate in train: {(train['stock_hour6_22_cnt'] > 0).mean() * 100:.1f}%")

del train_raw, ev_raw, so_rate
gc.collect()

### 1.2 Censored Demand Recovery (Proportional)

When stock runs out during the day, observed sales undercount true demand. We use a **proportional recovery** method:
- If stock was out for `k` of 16 operating hours, we scale sales by `16 / (16 - k)`
- Full stockout days use the rolling non-stockout average

In [ ]:
OP_HRS = 16
train = train.sort_values(['sp', 'dt']).reset_index(drop=True)
train['cens'] = (train['stock_hour6_22_cnt'] > 0).astype('int8')
train['so_frac'] = (train['stock_hour6_22_cnt'] / OP_HRS).astype('float32')
train['demand'] = train['sale_amount'].values.copy().astype(np.float64)

# Proportional recovery for partial stockouts
partial = (train['stock_hour6_22_cnt'] > 0) & (train['stock_hour6_22_cnt'] < OP_HRS)
avail = (OP_HRS - train['stock_hour6_22_cnt']).clip(lower=1)
train.loc[partial, 'demand'] = train.loc[partial, 'sale_amount'] * (OP_HRS / avail[partial])

# Full stockout: rolling non-stockout mean
full = train['stock_hour6_22_cnt'] >= OP_HRS
no_so = (train['cens'] == 0).astype(float)

train['_s'] = train['demand'] * no_so
g = train.groupby('sp')
rs = g['_s'].transform(lambda x: x.rolling(14, min_periods=1).sum())
rc = g.apply(lambda grp: no_so.loc[grp.index].rolling(14, min_periods=1).sum())
rc = rc.reset_index(level=0, drop=True).sort_index()
avg_ns = rs / rc.clip(lower=1)
train.loc[full, 'demand'] = avg_ns[full]

train['demand'] = train['demand'].clip(lower=0).astype('float32')
train.drop('_s', axis=1, inplace=True)

print(f"Mean sales:  {train['sale_amount'].mean():.4f}")
print(f"Mean demand: {train['demand'].mean():.4f}  "
      f"(+{(train['demand'].mean() / train['sale_amount'].mean() - 1) * 100:.1f}% recovery)")

del rs, rc, avg_ns
gc.collect()

### 1.3 Feature Engineering

Full pipeline: temporal, lags, rolling stats, EWMA, stockout features, cross features, global stats, hierarchy means, and KMeans clustering (k=25).

In [ ]:
def make_features(df):
    """Full feature engineering pipeline."""
    df = df.sort_values(['sp', 'dt']).reset_index(drop=True)
    t = 'demand'
    g = df.groupby('sp')

    # --- Temporal ---
    df['dow'] = df['dt'].dt.dayofweek.astype('int8')
    df['dom'] = df['dt'].dt.day.astype('int8')
    df['woy'] = df['dt'].dt.isocalendar().week.astype('int8')
    df['month'] = df['dt'].dt.month.astype('int8')
    df['wknd'] = (df['dow'] >= 5).astype('int8')
    df['doy'] = df['dt'].dt.dayofyear.astype('int16')
    df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7).astype('float32')
    df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7).astype('float32')
    df['dom_sin'] = np.sin(2 * np.pi * df['dom'] / 31).astype('float32')
    df['dom_cos'] = np.cos(2 * np.pi * df['dom'] / 31).astype('float32')
    df['woy_sin'] = np.sin(2 * np.pi * df['woy'] / 52).astype('float32')
    df['woy_cos'] = np.cos(2 * np.pi * df['woy'] / 52).astype('float32')

    # --- Lags ---
    for lag in [1, 2, 3, 5, 7, 14, 21, 28]:
        df[f'lag_{lag}'] = g[t].shift(lag).astype('float32')
    df['slag_1'] = g['sale_amount'].shift(1).astype('float32')
    df['slag_7'] = g['sale_amount'].shift(7).astype('float32')
    df['diff_1'] = (df['lag_1'] - g[t].shift(2)).astype('float32')
    df['diff_7'] = (df['lag_1'] - df['lag_7']).astype('float32')

    # --- Rolling stats ---
    shifted = g[t].shift(1)
    for w in [3, 7, 14, 28]:
        r = shifted.groupby(df['sp']).rolling(w, min_periods=1)
        df[f'rm_{w}'] = r.mean().reset_index(level=0, drop=True).astype('float32')
        df[f'rs_{w}'] = r.std().reset_index(level=0, drop=True).astype('float32')
        if w in [7, 28]:
            df[f'rx_{w}'] = r.max().reset_index(level=0, drop=True).astype('float32')
            df[f'rn_{w}'] = r.min().reset_index(level=0, drop=True).astype('float32')
            df[f'rmd_{w}'] = r.median().reset_index(level=0, drop=True).astype('float32')
        gc.collect()

    for w in [14, 28]:
        r = shifted.groupby(df['sp']).rolling(w, min_periods=1)
        df[f'rq25_{w}'] = r.quantile(0.25).reset_index(level=0, drop=True).astype('float32')
        df[f'rq75_{w}'] = r.quantile(0.75).reset_index(level=0, drop=True).astype('float32')

    # --- EWMA ---
    for s in [7, 14, 28]:
        df[f'ew_{s}'] = shifted.groupby(df['sp']).transform(
            lambda x: x.ewm(span=s, min_periods=1).mean()
        ).astype('float32')
    del shifted
    gc.collect()

    # --- Stockout features ---
    cs = g['cens'].shift(1)
    df['so1'] = cs.astype('float32')
    df['so7'] = g['cens'].shift(7).astype('float32')
    for w in [7, 14]:
        df[f'sor_{w}'] = cs.groupby(df['sp']).rolling(w, min_periods=1).mean() \
                           .reset_index(level=0, drop=True).astype('float32')
    hs = g['stock_hour6_22_cnt'].shift(1)
    for w in [7, 14]:
        df[f'soh_{w}'] = hs.groupby(df['sp']).rolling(w, min_periods=1).mean() \
                           .reset_index(level=0, drop=True).astype('float32')
    del cs, hs
    gc.collect()

    # --- Variability & trend ---
    df['cv7'] = (df['rs_7'] / df['rm_7'].clip(lower=0.01)).astype('float32')
    df['cv28'] = (df['rs_28'] / df['rm_28'].clip(lower=0.01)).astype('float32')
    df['tr_7_28'] = (df['rm_7'] - df['rm_28']).astype('float32')
    df['tr_3_14'] = (df['rm_3'] - df['rm_14']).astype('float32')
    df['lag1_rm7'] = (df['lag_1'] / df['rm_7'].clip(lower=0.01)).astype('float32')
    df['rm7_rm28'] = (df['rm_7'] / df['rm_28'].clip(lower=0.01)).astype('float32')

    # --- DOW profile ---
    df['dow_prof'] = df.groupby(['sp', 'dow'])[t].transform('mean').astype('float32')

    # --- Cross features ---
    df['d_h'] = (df['discount'] * df['holiday_flag']).astype('float32')
    df['d_a'] = (df['discount'] * df['activity_flag']).astype('float32')
    df['t_p'] = (df['avg_temperature'] * df['precpt']).astype('float32')
    df['w_h'] = (df['wknd'] * df['holiday_flag']).astype('int8')
    df['h_a'] = (df['holiday_flag'] * df['activity_flag']).astype('int8')
    df['temp_dev'] = (df['avg_temperature'] - df.groupby('sp')['avg_temperature'].transform('mean')).astype('float32')
    df['hum_dev'] = (df['avg_humidity'] - df.groupby('sp')['avg_humidity'].transform('mean')).astype('float32')

    # --- Global stats ---
    for grp, pfx in [('sp', 'sp'), ('product_id', 'pd'), ('store_id', 'st'), ('city_id', 'ct')]:
        df[f'{pfx}_m'] = df.groupby(grp)[t].transform('mean').astype('float32')
        df[f'{pfx}_s'] = df.groupby(grp)[t].transform('std').fillna(0).astype('float32')

    # --- Hierarchy means ---
    df['cat1_m'] = df.groupby('first_category_id')[t].transform('mean').astype('float32')
    df['cat2_m'] = df.groupby('second_category_id')[t].transform('mean').astype('float32')

    # --- KMeans clustering (k=25) ---
    sp_stats = df.groupby('sp').agg(
        mean_d=(t, 'mean'), std_d=(t, 'std'), max_d=(t, 'max'),
        so_rate=('cens', 'mean')
    ).fillna(0)
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    sp_scaled = scaler.fit_transform(sp_stats)
    km = KMeans(n_clusters=25, random_state=42, n_init=10)
    sp_stats['cluster'] = km.fit_predict(sp_scaled)
    # Drop pre-existing cluster column if re-running make_features
    if 'cluster' in df.columns:
        df.drop('cluster', axis=1, inplace=True)
    df = df.merge(sp_stats[['cluster']], left_on='sp', right_index=True, how='left')
    df['cluster'] = df['cluster'].astype('int8')

    df = df.fillna(0)
    gc.collect()
    return df


def get_fcols(df):
    """Return feature columns (exclude target and identifiers)."""
    excl = {'dt', 'sale_amount', 'demand', 'sp', 'cens', 'so_frac'}
    return [c for c in df.columns if c not in excl]


print("Building features on training data...")
train = make_features(train)
fcols = get_fcols(train)
print(f"Features: {len(fcols)}, Shape: {train.shape}")
print(f"Memory: {train.memory_usage(deep=True).sum() / 1e6:.0f} MB")

### 1.4 Train LightGBM Models

We train five models:
- **MAE** and **Huber**: robust point forecasts (with censored weight = 0.5)
- **Q10, Q50, Q90**: quantile regression for uncertainty bounds

Validation: last 7 days of training data.

In [ ]:
LGB_BASE = {
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'learning_rate': 0.05,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.75,
    'bagging_freq': 5,
    'min_child_samples': 30,
    'lambda_l1': 0.1,
    'lambda_l2': 1.0,
    'verbose': -1,
    'n_jobs': -1,
    'seed': 42,
}

# Time-based split: last 7 days for validation
val_cut = train['dt'].max() - pd.Timedelta(days=6)
warmup = train['dt'].min() + pd.Timedelta(days=28)
tr_mask = (train['dt'] < val_cut) & (train['dt'] >= warmup)
vl_mask = train['dt'] >= val_cut

Xtr, ytr = train.loc[tr_mask, fcols].values, train.loc[tr_mask, 'demand'].values
Xvl, yvl = train.loc[vl_mask, fcols].values, train.loc[vl_mask, 'demand'].values

# Censored weight = 0.5 for MAE/Huber
wtr = np.ones(len(ytr), dtype='float32')
wtr[train.loc[tr_mask, 'cens'].values == 1] = 0.5

print(f"Train: {len(ytr):,}  |  Val: {len(yvl):,}  |  Features: {len(fcols)}")
print(f"Val period: {val_cut.date()} to {train['dt'].max().date()}")

models = {}

# --- MAE model ---
print("\n--- Training MAE model ---")
dtrain = lgb.Dataset(Xtr, ytr, weight=wtr, feature_name=fcols)
dval = lgb.Dataset(Xvl, yvl, feature_name=fcols, reference=dtrain)
params_mae = {**LGB_BASE, 'objective': 'regression_l1', 'metric': 'mae'}
models['mae'] = lgb.train(
    params_mae, dtrain, 800,
    valid_sets=[dtrain, dval], valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(500)]
)
print(f"  Best iteration: {models['mae'].best_iteration}")

# --- Huber model ---
print("\n--- Training Huber model ---")
params_huber = {**LGB_BASE, 'objective': 'huber', 'metric': 'mae', 'huber_delta': 1.0}
models['huber'] = lgb.train(
    params_huber, dtrain, 800,
    valid_sets=[dtrain, dval], valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(500)]
)
print(f"  Best iteration: {models['huber'].best_iteration}")

# --- Quantile models (no censored weights) ---
dtrain_q = lgb.Dataset(Xtr, ytr, feature_name=fcols)
dval_q = lgb.Dataset(Xvl, yvl, feature_name=fcols, reference=dtrain_q)

for q, name in [(0.1, 'q10'), (0.5, 'q50'), (0.9, 'q90')]:
    print(f"\n--- Training {name.upper()} model (alpha={q}) ---")
    params_q = {**LGB_BASE, 'objective': 'quantile', 'alpha': q, 'metric': 'quantile'}
    models[name] = lgb.train(
        params_q, dtrain_q, 800,
        valid_sets=[dtrain_q, dval_q], valid_names=['train', 'val'],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(500)]
    )
    print(f"  Best iteration: {models[name].best_iteration}")

# Validation summary
print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)
for name in ['mae', 'huber', 'q50']:
    p = np.clip(models[name].predict(Xvl), 0, None)
    mae_val = np.mean(np.abs(yvl - p))
    print(f"  {name.upper():8s}  MAE = {mae_val:.4f}")

p_ens = np.clip((models['mae'].predict(Xvl) + models['huber'].predict(Xvl) + models['q50'].predict(Xvl)) / 3, 0, None)
print(f"  {'ENSEMBLE':8s}  MAE = {np.mean(np.abs(yvl - p_ens)):.4f}")

p10_vl = np.clip(models['q10'].predict(Xvl), 0, None)
p90_vl = np.clip(models['q90'].predict(Xvl), 0, None)
cov = np.mean((yvl >= p10_vl) & (yvl <= p90_vl))
print(f"\n  80% PI coverage (Q10-Q90): {cov * 100:.1f}%")

del Xtr, ytr, Xvl, yvl, wtr, dtrain, dval, dtrain_q, dval_q
gc.collect()

### 1.5 Generate Eval Predictions & Historical Std

In [ ]:
# Prepare eval data: combine recent history + eval for feature computation
ev['cens'] = (ev['stock_hour6_22_cnt'] > 0).astype('int8')
ev['so_frac'] = (ev['stock_hour6_22_cnt'] / 16).astype('float32')
ev['demand'] = ev['sale_amount'].astype('float32')

hist = train[train['dt'] >= train['dt'].max() - pd.Timedelta(days=27)].copy()
comb = pd.concat([hist, ev], ignore_index=True)
del hist
gc.collect()

print("Building features on eval data...")
comb = make_features(comb)

eval_dates = sorted(ev['dt'].unique())
emask = comb['dt'].isin(eval_dates)
ef = comb[emask].copy()

for c in fcols:
    if c not in ef.columns:
        ef[c] = 0

X_eval = ef[fcols].fillna(0).values
y_eval = ef['sale_amount'].values
sp_eval = ef['sp'].values

# Generate predictions
preds = {}
for name, mdl in models.items():
    preds[name] = np.clip(mdl.predict(X_eval), 0, None)

# Ensemble = (MAE + Huber + Q50) / 3
preds['ensemble'] = np.clip((preds['mae'] + preds['huber'] + preds['q50']) / 3, 0, None)

# Per-SP historical std from training data
sp_hist_std = train.groupby('sp')['demand'].std().fillna(0.1).to_dict()

print(f"\nEval predictions: {len(y_eval):,} observations")
print(f"Ensemble MAE: {np.mean(np.abs(y_eval - preds['ensemble'])):.4f}")
print(f"80% PI coverage: {np.mean((y_eval >= preds['q10']) & (y_eval <= preds['q90'])) * 100:.1f}%")

# Store key arrays for inventory optimization
mu = preds['ensemble']
q10_pred = preds['q10']
q50_pred = preds['q50']
q90_pred = preds['q90']
D = y_eval  # actual demand (observed sales)

del comb, ef
gc.collect()
print("Forecasts ready for inventory optimization.")

---
## Section 2: The Newsvendor Problem

The **newsvendor problem** (also called the newsboy problem) is the foundational model for inventory decisions under uncertainty. It captures the core tension: **order too much and waste money; order too little and lose sales.**

### Problem Statement

A retailer must decide **how many units Q** to order for a perishable product **before observing demand D**.

- If $Q > D$: we have **overage** of $Q - D$ units. Each excess unit incurs:
  - **Holding cost** $h$ (storage, refrigeration)
  - **Waste cost** $w$ (spoiled perishable goods)
- If $Q < D$: we have **underage** of $D - Q$ units. Each unmet unit incurs:
  - **Stockout penalty** $p$ (lost profit, customer dissatisfaction, lost future sales)

### The Optimal Solution: Critical Ratio

The optimal order quantity minimizes expected total cost:

$$\min_Q \; \mathbb{E}\left[(h + w) \cdot (Q - D)^+ + p \cdot (D - Q)^+\right]$$

Taking the derivative and setting to zero yields the **critical ratio**:

$$CR = \frac{p}{p + h + w}$$

The optimal $Q^*$ satisfies $F(Q^*) = CR$, where $F$ is the CDF of demand.

### Under the Normal Assumption

If $D \sim \mathcal{N}(\mu, \sigma^2)$:

$$Q^* = \mu + \Phi^{-1}(CR) \cdot \sigma$$

where $\Phi^{-1}$ is the standard normal inverse CDF (quantile function).

In [ ]:
# Define cost configuration
cfg = {
    'h': 0.10,            # holding cost per unit
    'p': 0.50,            # stockout penalty per unit
    'w': 0.30,            # waste cost per unit (perishable)
    'unit_revenue': 1.0,  # revenue per unit sold
    'unit_cost': 0.40,    # procurement cost per unit
    'shelf_life': 3,      # days before expiry
    'lead_time': 1,       # days from order to delivery
    'alpha': 0.95,        # target service level
}

# Critical ratio
CR = cfg['p'] / (cfg['p'] + cfg['h'] + cfg['w'])
z_cr = norm.ppf(CR)

print("=" * 50)
print("NEWSVENDOR COST CONFIGURATION")
print("=" * 50)
print(f"  Holding cost  (h):  {cfg['h']:.2f} per excess unit")
print(f"  Stockout cost (p):  {cfg['p']:.2f} per unmet unit")
print(f"  Waste cost    (w):  {cfg['w']:.2f} per excess unit")
print(f"  Overage cost (h+w): {cfg['h'] + cfg['w']:.2f}")
print(f"  Unit revenue:       {cfg['unit_revenue']:.2f}")
print(f"  Unit cost:          {cfg['unit_cost']:.2f}")
print(f"  Shelf life:         {cfg['shelf_life']} days")
print(f"  Lead time:          {cfg['lead_time']} day")
print()
print(f"  Critical Ratio (CR) = p / (p + h + w)")
print(f"                      = {cfg['p']:.2f} / ({cfg['p']:.2f} + {cfg['h']:.2f} + {cfg['w']:.2f})")
print(f"                      = {CR:.4f}")
print(f"  z(CR) = Phi^-1({CR:.4f}) = {z_cr:.4f}")
print()
print(f"  Interpretation: CR = {CR:.3f} > 0.5")
print(f"  --> Stockout is MORE costly than overage.")
print(f"  --> Optimal order is ABOVE the mean (z = {z_cr:.3f} > 0).")
print(f"  --> We should carry some safety stock.")

### Intuition Behind the Critical Ratio

| CR Value | Meaning | Order Decision |
|----------|---------|----------------|
| CR < 0.5 | Overage more costly | Order **below** the mean |
| CR = 0.5 | Symmetric costs | Order at the **mean** |
| CR > 0.5 | Stockout more costly | Order **above** the mean |
| CR -> 1.0 | Stockout hugely costly | Order near **max** demand |

In our case, $CR = 0.625$ means each lost sale costs \$0.50, while each excess unit costs only \$0.40 (holding + waste). The asymmetry pushes us to order slightly above expected demand.

---
## Section 3: The Gaussian Assumption Problem

The standard newsvendor solution $Q^* = \mu + \Phi^{-1}(CR) \cdot \sigma$ assumes demand follows a **Normal distribution**. In practice, this assumption is often violated for fresh retail demand.

### Why Normal Fails for Retail Demand

1. **Negative demand is impossible**: The Normal distribution has support on $(-\infty, +\infty)$, but demand $\geq 0$ always.

2. **Right-skewed demand**: Fresh products often have occasional high-demand days (promotions, holidays) creating a long right tail.

3. **Zero-inflated demand**: Many store-products have days with zero sales (slow movers, out of season, stockouts). This creates a mass at zero that the Normal cannot capture.

4. **Estimated uncertainty**: The $\sigma$ we use is estimated from historical data, introducing additional error. The Normal assumes $\sigma$ is known exactly.

In [ ]:
# Compute sigma: blend of historical std and residual std
sigma_hist = np.array([max(sp_hist_std.get(k, 0.1), 0.05) for k in sp_eval])
residual_std = np.std(mu - D)
sigma = np.sqrt(0.5 * sigma_hist**2 + 0.5 * residual_std**2)

print(f"Residual std (global):  {residual_std:.4f}")
print(f"Historical std (mean):  {np.mean(sigma_hist):.4f}")
print(f"Blended sigma (mean):   {np.mean(sigma):.4f}")

In [ ]:
# Visualize the Normal assumption failure for 4 example SPs
unique_sps = np.unique(sp_eval)

# Select 4 SPs with diverse patterns
sp_demand_chars = []
for sp in unique_sps:
    sp_mask_train = train['sp'] == sp
    if sp_mask_train.sum() < 30:
        continue
    d = train.loc[sp_mask_train, 'demand'].values
    sp_demand_chars.append({
        'sp': sp,
        'mean': np.mean(d),
        'std': np.std(d),
        'zero_frac': np.mean(d < 0.5),
        'skew': float(stats.skew(d)),
        'data': d
    })

sp_demand_chars.sort(key=lambda x: x['mean'])

# Pick 4: low-demand/zero-inflated, moderate, high, right-skewed
example_sps = []
# 1. Zero-inflated (highest zero fraction)
zero_inflated = sorted(sp_demand_chars, key=lambda x: -x['zero_frac'])
example_sps.append(zero_inflated[0])
# 2. Right-skewed (highest skewness, exclude zero-inflated)
skewed = sorted([s for s in sp_demand_chars if s['zero_frac'] < 0.3], key=lambda x: -x['skew'])
if skewed:
    example_sps.append(skewed[0])
# 3. Moderate demand (near median mean)
mid_idx = len(sp_demand_chars) // 2
example_sps.append(sp_demand_chars[mid_idx])
# 4. High demand (near top)
example_sps.append(sp_demand_chars[-max(1, len(sp_demand_chars)//20)])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Normal Assumption vs Actual Demand Distribution', fontsize=14, fontweight='bold')

labels = ['Zero-Inflated', 'Right-Skewed', 'Moderate', 'High Demand']
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

for idx, (sp_info, ax, label, color) in enumerate(zip(example_sps, axes.flat, labels, colors)):
    d = sp_info['data']
    mu_sp, sigma_sp = np.mean(d), np.std(d)
    
    # Histogram of actual demand
    bins = np.arange(max(0, d.min() - 0.5), d.max() + 1.5, max(1, (d.max() - d.min()) / 30))
    ax.hist(d, bins=bins, density=True, alpha=0.6, color=color, edgecolor='white',
            label=f'Actual (n={len(d)})')
    
    # Fitted Normal PDF
    x_range = np.linspace(max(-3 * sigma_sp + mu_sp, -1), mu_sp + 4 * sigma_sp, 200)
    normal_pdf = norm.pdf(x_range, mu_sp, max(sigma_sp, 0.01))
    ax.plot(x_range, normal_pdf, 'k--', lw=2, label=f'Normal({mu_sp:.1f}, {sigma_sp:.1f})')
    
    # Shade the negative region (impossible)
    neg_x = x_range[x_range < 0]
    if len(neg_x) > 0:
        neg_pdf = norm.pdf(neg_x, mu_sp, max(sigma_sp, 0.01))
        ax.fill_between(neg_x, neg_pdf, alpha=0.3, color='red', label='Negative (impossible)')
    
    ax.set_title(f'{label} (SP={sp_info["sp"]})', fontweight='bold')
    ax.set_xlabel('Demand')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.text(0.97, 0.97,
            f'mean={mu_sp:.1f}\nstd={sigma_sp:.1f}\nzero%={sp_info["zero_frac"]*100:.0f}%\nskew={sp_info["skew"]:.1f}',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb05_normal_assumption_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observations:")
print("  - Zero-inflated SP: large mass at 0 that Normal cannot model")
print("  - Right-skewed SP: long right tail, Normal underestimates high-demand events")
print("  - The Normal PDF assigns probability to negative demand (physically impossible)")
print("  - These mismatches propagate into suboptimal ordering decisions")

---
## Section 4: Inventory Policies -- Normal-Based

We now implement and evaluate a range of inventory policies, starting with those that assume (or approximate) Normal demand.

### Evaluation Function

Each policy produces an order quantity $Q$ for each observation. We measure:
- **Service Level (Type 1)**: fraction of periods where $Q \geq D$ (no stockout)
- **Fill Rate**: fraction of total demand satisfied ($\sum \min(Q,D) / \sum D$)
- **Average Profit**: revenue minus all costs
- **Stockout Rate**: fraction of periods with unmet demand
- **Waste %**: fraction of ordered units that go unsold

In [ ]:
def eval_policy(Q, D, cfg, name=""):
    """Evaluate an inventory policy on actual demand."""
    Q, D = np.asarray(Q, dtype=float), np.asarray(D, dtype=float)
    sold = np.minimum(Q, D)
    over = np.maximum(Q - D, 0)
    under = np.maximum(D - Q, 0)
    revenue = sold * cfg['unit_revenue']
    procurement = Q * cfg['unit_cost']
    holding = cfg['h'] * over
    stockout = cfg['p'] * under
    waste = cfg['w'] * over
    total_cost = procurement + holding + stockout + waste
    profit = revenue - total_cost
    return {
        'Policy': name,
        'SL(Type1)': round(np.mean(Q >= D), 4),
        'Fill Rate': round(np.sum(sold) / max(np.sum(D), 1), 4),
        'Avg Profit': round(np.mean(profit), 4),
        'SO Rate': round(np.mean(D > Q), 4),
        'Waste%': round(np.sum(over) / max(np.sum(Q), 1) * 100, 2),
    }

print("eval_policy() defined.")

### 4.1 Newsvendor (Normal)

$Q^* = \max\left(\mu + \Phi^{-1}(CR) \cdot \sigma, \; 0\right)$

### 4.2 Quantile Newsvendor

Uses the quantile predictions directly: interpolate between Q50 and Q90 based on CR.

$Q = Q_{50} + \frac{CR - 0.5}{0.9 - 0.5} \cdot (Q_{90} - Q_{50})$

### 4.3 Service Level ($\alpha = 0.95$)

Ignore cost trade-offs; order to achieve a target service level.

$Q = \max\left(\mu + \Phi^{-1}(0.95) \cdot \sigma, \; 0\right)$

### 4.4 SAA (Sample Average Approximation)

Generate 2000 demand scenarios from $\mathcal{N}(\mu, \sigma^2)$, then take the CR quantile.

### 4.5 Perishable Cost Optimization

Adjust the critical ratio for shelf life: shorter shelf life means higher effective waste cost.

### 4.6 Dynamic Safety Stock

Adapt the safety factor $z$ based on coefficient of variation (CV):
- Low CV (< 0.3): $z = 0.5$ (stable demand, less safety stock needed)
- Medium CV (0.3--0.7): $z = 1.0$
- High CV (> 0.7): $z = 1.5$ (volatile demand, more safety stock)

In [ ]:
results = []

# --- Policy 1: Newsvendor (Normal) ---
Q_nv = np.maximum(mu + z_cr * sigma, 0)
results.append(eval_policy(Q_nv, D, cfg, f"Newsvendor (Normal, CR={CR:.3f})"))

# --- Policy 2: Quantile Newsvendor ---
t_interp = (CR - 0.5) / (0.9 - 0.5)
Q_qnv = np.maximum(q50_pred + t_interp * (q90_pred - q50_pred), 0)
results.append(eval_policy(Q_qnv, D, cfg, "Quantile Newsvendor"))

# --- Policy 3: Service Level alpha=0.95 ---
z_sl = norm.ppf(cfg['alpha'])
Q_sl = np.maximum(mu + z_sl * sigma, 0)
results.append(eval_policy(Q_sl, D, cfg, f"Service Level (alpha={cfg['alpha']})"))

# --- Policy 4: SAA (2000 scenarios) ---
np.random.seed(42)
n_scenarios = 2000
scenarios = np.random.normal(
    mu[:, None], sigma[:, None], (len(mu), n_scenarios)
)
scenarios = np.clip(scenarios, 0, None)
Q_saa = np.quantile(scenarios, CR, axis=1)
results.append(eval_policy(Q_saa, D, cfg, f"SAA ({n_scenarios} scenarios)"))
del scenarios
gc.collect()

# --- Policy 5: Perishable Cost Opt ---
waste_factor = max(0, 1 - cfg['shelf_life'] * 0.3)
co_eff = cfg['h'] + cfg['w'] * waste_factor
cr_perish = cfg['p'] / (cfg['p'] + co_eff)
z_perish = norm.ppf(cr_perish)
Q_perish = np.maximum(mu + z_perish * sigma, 0)
results.append(eval_policy(Q_perish, D, cfg, f"Perishable Cost Opt (CR={cr_perish:.3f})"))

# --- Policy 6: Dynamic Safety Stock ---
cv = sigma / np.maximum(mu, 0.01)
z_dyn = np.where(cv < 0.3, 0.5, np.where(cv < 0.7, 1.0, 1.5))
Q_dss = np.maximum(mu + z_dyn * sigma, 0)
results.append(eval_policy(Q_dss, D, cfg, "Dynamic Safety Stock"))

# --- Policy 7: Q50 Direct ---
results.append(eval_policy(np.maximum(q50_pred, 0), D, cfg, "Q50 Direct"))

# --- Policy 8: Q90 Direct ---
results.append(eval_policy(np.maximum(q90_pred, 0), D, cfg, "Q90 Direct"))

# --- Policy 9: Naive (Q = forecast) ---
results.append(eval_policy(np.maximum(mu, 0), D, cfg, "Naive (Q=forecast)"))

# --- Policies 10-14: Safety Stock Variants ---
for ss_mult in [0.25, 0.5, 0.75, 1.0, 1.5]:
    Q_ss = np.maximum(mu + ss_mult * sigma, 0)
    results.append(eval_policy(Q_ss, D, cfg, f"Safety Stock ({ss_mult}*sigma)"))

# Display Normal-based results
df_results_normal = pd.DataFrame(results)
print("\n" + "=" * 90)
print("NORMAL-BASED INVENTORY POLICIES")
print("=" * 90)
print(df_results_normal.to_string(index=False))
df_results_normal.to_csv(os.path.join(NB_OUT, 'nb05_normal_policy_results.csv'), index=False)
print(f'Saved nb05_normal_policy_results.csv to {NB_OUT}')

---
## Section 5: Non-Normal Distribution Policies

The policies above all rely on the Normal distribution (or quantile predictions from a model). Here we explore methods that make **fewer distributional assumptions**, which is especially important for the types of demand patterns we identified in Section 3.

### 5.1 Empirical Newsvendor

**Idea**: Instead of assuming demand follows any parametric distribution, use the **actual forecast residuals** (actual - predicted) from the training set. The CR quantile of these residuals becomes the safety stock.

$$Q = \hat{\mu} + \text{quantile}(\text{residuals}, CR)$$

This approach bypasses all distribution assumptions. It works well when the forecast model is stable and residuals are representative.

### 5.2 Zero-Inflated Gamma

**Idea**: Model demand as a mixture:
- With probability $p_0$: demand = 0
- With probability $1 - p_0$: demand ~ Gamma(shape, scale)

The Gamma distribution is non-negative and right-skewed, making it a better fit for retail demand.

To find the CR quantile:
1. If $CR \leq p_0$: order 0 (likely zero demand)
2. If $CR > p_0$: adjust $CR_{adj} = (CR - p_0)/(1 - p_0)$ and use Gamma quantile

### 5.3 KDE Newsvendor

**Idea**: Use **kernel density estimation** on historical demand to get a fully non-parametric demand distribution. Then numerically find the CR quantile of this distribution.

### 5.4 Quantile-Direct Interpolated

**Idea**: Use the model's quantile predictions (Q10, Q50, Q90) directly and interpolate to match any target quantile. This is model-free in the sense that it does not assume any distributional form on demand -- it relies entirely on the learned quantile model.

In [ ]:
# ---- Policy: Empirical Newsvendor ----
# Compute per-SP residuals on training data, then find CR quantile as safety stock

# First, get training predictions for residual computation
warmup_date = train['dt'].min() + pd.Timedelta(days=28)
train_pred_mask = train['dt'] >= warmup_date
X_train_all = train.loc[train_pred_mask, fcols].fillna(0).values
y_train_all = train.loc[train_pred_mask, 'demand'].values
sp_train_all = train.loc[train_pred_mask, 'sp'].values

# Ensemble prediction on training data
mu_train = np.clip(
    (models['mae'].predict(X_train_all) +
     models['huber'].predict(X_train_all) +
     models['q50'].predict(X_train_all)) / 3, 0, None
)
residuals_train = y_train_all - mu_train

# Per-SP empirical CR quantile of residuals
sp_residual_quantile = {}
for sp in np.unique(sp_train_all):
    sp_mask = sp_train_all == sp
    res = residuals_train[sp_mask]
    if len(res) >= 5:
        sp_residual_quantile[sp] = np.quantile(res, CR)
    else:
        sp_residual_quantile[sp] = np.quantile(residuals_train, CR)  # fallback: global

# Apply empirical safety stock
empirical_ss = np.array([sp_residual_quantile.get(sp, 0) for sp in sp_eval])
Q_empirical = np.maximum(mu + empirical_ss, 0)

results.append(eval_policy(Q_empirical, D, cfg, "Empirical Newsvendor"))
print("Empirical Newsvendor: uses per-SP residual quantile as safety stock")
print(f"  Mean empirical safety stock: {np.mean(empirical_ss):.4f}")
print(f"  Std of empirical safety stock: {np.std(empirical_ss):.4f}")

del X_train_all
gc.collect()

In [ ]:
# ---- Policy: Zero-Inflated Gamma ----
# Fit Gamma to positive demand for each SP, then compute adjusted CR quantile

Q_zig = np.zeros(len(mu))
zig_fit_count = 0
zig_fallback_count = 0

for sp in np.unique(sp_eval):
    # Historical demand for this SP
    sp_mask_train = train['sp'] == sp
    d_hist = train.loc[sp_mask_train, 'demand'].values
    
    # Eval mask
    sp_mask_eval = sp_eval == sp
    
    if len(d_hist) < 10:
        # Fallback to Normal newsvendor
        Q_zig[sp_mask_eval] = np.maximum(mu[sp_mask_eval] + z_cr * sigma[sp_mask_eval], 0)
        zig_fallback_count += 1
        continue
    
    # Zero probability
    p0 = np.mean(d_hist < 0.5)
    positive_demand = d_hist[d_hist >= 0.5]
    
    if len(positive_demand) < 5 or CR <= p0:
        # If CR <= p0, optimal order is 0 (mostly zero demand)
        Q_zig[sp_mask_eval] = 0 if CR <= p0 else np.maximum(mu[sp_mask_eval], 0)
        zig_fallback_count += 1
        continue
    
    try:
        # Fit Gamma to positive demand
        shape, _, scale = gamma.fit(positive_demand, floc=0)
        # Adjusted critical ratio
        adj_cr = (CR - p0) / (1 - p0)
        adj_cr = np.clip(adj_cr, 0.01, 0.99)
        q_val = gamma.ppf(adj_cr, shape, scale=scale)
        Q_zig[sp_mask_eval] = np.maximum(q_val, 0)
        zig_fit_count += 1
    except Exception:
        # Fallback to Normal
        Q_zig[sp_mask_eval] = np.maximum(mu[sp_mask_eval] + z_cr * sigma[sp_mask_eval], 0)
        zig_fallback_count += 1

results.append(eval_policy(Q_zig, D, cfg, "Zero-Inflated Gamma"))
print(f"Zero-Inflated Gamma: {zig_fit_count} SPs fitted, {zig_fallback_count} fallbacks")

In [ ]:
# ---- Policy: KDE Newsvendor ----
# Fully non-parametric: kernel density estimation on historical demand

from scipy.stats import gaussian_kde

Q_kde = np.zeros(len(mu))
kde_fit_count = 0
kde_fallback_count = 0

for sp in np.unique(sp_eval):
    sp_mask_train = train['sp'] == sp
    d_hist = train.loc[sp_mask_train, 'demand'].values
    sp_mask_eval = sp_eval == sp
    
    if len(d_hist) < 15 or np.std(d_hist) < 0.01:
        # Too little data or constant demand
        Q_kde[sp_mask_eval] = np.maximum(mu[sp_mask_eval] + z_cr * sigma[sp_mask_eval], 0)
        kde_fallback_count += 1
        continue
    
    try:
        kde = gaussian_kde(d_hist)
        # Numerically compute CR quantile via CDF on a grid
        x_grid = np.linspace(0, np.max(d_hist) * 1.5 + 1, 500)
        cdf_vals = np.array([kde.integrate_box_1d(0, x) for x in x_grid])
        # Find the grid point where CDF >= CR
        idx_cr = np.searchsorted(cdf_vals, CR)
        if idx_cr < len(x_grid):
            q_val = x_grid[idx_cr]
        else:
            q_val = x_grid[-1]
        Q_kde[sp_mask_eval] = np.maximum(q_val, 0)
        kde_fit_count += 1
    except Exception:
        Q_kde[sp_mask_eval] = np.maximum(mu[sp_mask_eval] + z_cr * sigma[sp_mask_eval], 0)
        kde_fallback_count += 1

results.append(eval_policy(Q_kde, D, cfg, "KDE Newsvendor"))
print(f"KDE Newsvendor: {kde_fit_count} SPs fitted, {kde_fallback_count} fallbacks")

In [ ]:
# ---- Policy: Quantile-Direct Interpolated ----
# Interpolate between Q10, Q50, Q90 to match any target quantile
# Same idea as Quantile Newsvendor but explicit about being model-free

target_q = CR  # we want the CR quantile

if target_q <= 0.10:
    Q_qdi = q10_pred
elif target_q <= 0.50:
    # Interpolate between Q10 and Q50
    t_val = (target_q - 0.10) / (0.50 - 0.10)
    Q_qdi = q10_pred + t_val * (q50_pred - q10_pred)
elif target_q <= 0.90:
    # Interpolate between Q50 and Q90
    t_val = (target_q - 0.50) / (0.90 - 0.50)
    Q_qdi = q50_pred + t_val * (q90_pred - q50_pred)
else:
    Q_qdi = q90_pred

Q_qdi = np.maximum(Q_qdi, 0)
results.append(eval_policy(Q_qdi, D, cfg, "Quantile-Direct Interp"))

print(f"Quantile-Direct Interpolated: target quantile = {target_q:.3f}")
print(f"  Interpolation between Q50 and Q90 with t = {(target_q - 0.50) / (0.90 - 0.50):.3f}")

---
## Section 6: Full Policy Comparison

Now let us bring all policies together and compare their performance.

In [ ]:
# Full results table
df_all = pd.DataFrame(results).sort_values('Avg Profit', ascending=False).reset_index(drop=True)

print("\n" + "=" * 100)
print("FULL INVENTORY POLICY COMPARISON (sorted by Avg Profit)")
print("=" * 100)
print(df_all.to_string(index=False))
print("\nTotal policies evaluated:", len(df_all))
df_all.to_csv(os.path.join(NB_OUT, 'nb05_policy_comparison.csv'), index=False)
print(f'Saved nb05_policy_comparison.csv to {NB_OUT}')

In [ ]:
# Identify which policies are non-Normal
non_normal_names = ['Empirical Newsvendor', 'Zero-Inflated Gamma', 'KDE Newsvendor',
                    'Quantile-Direct Interp', 'Quantile Newsvendor',
                    'Q50 Direct', 'Q90 Direct']

df_all['Type'] = df_all['Policy'].apply(
    lambda x: 'Non-Normal' if any(nn in x for nn in non_normal_names) else 'Normal-Based'
)

# ---- Pareto Frontier: Fill Rate vs Avg Total Cost ----
# Compute Avg Total Cost for the plot
# We re-compute costs for each policy from the results
# Avg Total Cost = unit_cost * avg_order + h * avg_overage + w * avg_overage + p * avg_underage
# Since we only stored summary metrics, approximate total cost from: Avg Profit = Avg Revenue - Avg Total Cost
# Avg Revenue = Fill Rate * Avg Demand * unit_revenue (approximately)
avg_demand = np.mean(D)
df_all['Approx Revenue'] = df_all['Fill Rate'] * avg_demand * cfg['unit_revenue']
df_all['Approx Total Cost'] = df_all['Approx Revenue'] - df_all['Avg Profit']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Pareto frontier plot ---
ax = axes[0]
for _, row in df_all.iterrows():
    color = '#e74c3c' if row['Type'] == 'Non-Normal' else '#3498db'
    if row['Fill Rate'] >= 0.95:
        color = '#f1c40f'  # gold for high-FR policies
    ax.scatter(row['Fill Rate'], row['Approx Total Cost'],
              s=100, c=color, edgecolors='k', linewidth=0.5, zorder=5)
    ax.annotate(row['Policy'][:18], (row['Fill Rate'], row['Approx Total Cost']),
               fontsize=6, ha='center', va='bottom', rotation=15)

ax.set_xlabel('Fill Rate', fontsize=12)
ax.set_ylabel('Approx Avg Total Cost', fontsize=12)
ax.set_title('Pareto Frontier: Fill Rate vs Cost', fontsize=13, fontweight='bold')
ax.axvline(0.95, color='gray', linestyle=':', lw=1, alpha=0.7, label='95% Fill Rate target')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=10, label='Normal-Based'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Non-Normal'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f1c40f', markersize=10, label='Fill Rate >= 95%'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='upper left')

# --- Profit bar chart: Top 12 ---
ax = axes[1]
top12 = df_all.nlargest(12, 'Avg Profit')
colors_bar = []
for _, row in top12.iterrows():
    if row['Type'] == 'Non-Normal':
        colors_bar.append('#e74c3c')
    elif row['Fill Rate'] >= 0.95:
        colors_bar.append('#f1c40f')
    else:
        colors_bar.append('#3498db')

bars = ax.barh(range(len(top12)), top12['Avg Profit'].values, color=colors_bar,
               edgecolor='k', linewidth=0.5)
ax.set_yticks(range(len(top12)))
ax.set_yticklabels([p[:25] for p in top12['Policy'].values], fontsize=8)
ax.set_xlabel('Avg Profit', fontsize=12)
ax.set_title('Top 12 Policies by Profit', fontsize=13, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, top12['Avg Profit'].values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

legend_elements_bar = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#3498db', markersize=10, label='Normal-Based'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#e74c3c', markersize=10, label='Non-Normal'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#f1c40f', markersize=10, label='FR >= 95%'),
]
ax.legend(handles=legend_elements_bar, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb05_pareto_frontier.png'), dpi=150, bbox_inches='tight')
plt.show()

### Discussion: Trade-offs

Key observations from the comparison:

1. **Highest profit does not always mean highest fill rate.** A policy that slightly under-orders can avoid waste costs and achieve better profit, even if it loses a few sales.

2. **Non-Normal policies** (red) can outperform Normal-based ones when demand deviates from Gaussian assumptions. The empirical and quantile-based methods are particularly robust because they make no distributional assumption.

3. **High fill rate comes at a cost.** The gold-colored policies with FR >= 95% tend to have more waste. Whether this is acceptable depends on the business context:
   - **Supermarket staple**: high service level required (customers leave if key items are missing)
   - **Specialty item**: lower fill rate acceptable (customers may substitute)

4. **The Pareto frontier** shows the efficient trade-off boundary. Policies below-left of the frontier are dominated -- another policy achieves the same fill rate at lower cost.

---
## Section 7: Sensitivity Analysis

How sensitive is the optimal order to the **stockout cost** $p$? This is crucial because:
- $p$ is often hard to estimate precisely (how much is a lost customer worth?)
- Small changes in $p$ can shift the critical ratio significantly
- The business decision should be robust to reasonable variations in cost parameters

We vary $p$ from 0.20 to 1.00 while holding $h = 0.10$ and $w = 0.30$ fixed.

In [ ]:
p_values = np.arange(0.20, 1.05, 0.05)
sensitivity_results = []

for p_val in p_values:
    cfg_temp = cfg.copy()
    cfg_temp['p'] = p_val
    
    # Compute optimal CR
    cr_temp = p_val / (p_val + cfg['h'] + cfg['w'])
    z_temp = norm.ppf(cr_temp)
    
    # Normal Newsvendor order
    Q_temp = np.maximum(mu + z_temp * sigma, 0)
    result = eval_policy(Q_temp, D, cfg_temp, f"p={p_val:.2f}")
    result['p'] = p_val
    result['CR'] = cr_temp
    result['z'] = z_temp
    result['Avg Order'] = np.mean(Q_temp)
    sensitivity_results.append(result)

df_sens = pd.DataFrame(sensitivity_results)
print("Sensitivity to stockout cost p:")
print(df_sens[['p', 'CR', 'z', 'Avg Order', 'Fill Rate', 'Waste%', 'Avg Profit', 'SO Rate']].to_string(index=False))
df_sens.to_csv(os.path.join(NB_OUT, 'nb05_sensitivity_analysis.csv'), index=False)
print(f'Saved nb05_sensitivity_analysis.csv to {NB_OUT}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sensitivity Analysis: How Stockout Cost Shapes Inventory Decisions',
             fontsize=14, fontweight='bold')

# 1. CR vs p
ax = axes[0, 0]
ax.plot(df_sens['p'], df_sens['CR'], 'o-', color='#2c3e50', lw=2, markersize=6)
ax.axhline(0.5, color='gray', linestyle=':', lw=1, label='CR = 0.5 (symmetric)')
ax.fill_between(df_sens['p'], 0.5, df_sens['CR'],
                where=df_sens['CR'] >= 0.5, alpha=0.15, color='green', label='Order above mean')
ax.set_xlabel('Stockout Cost (p)', fontsize=11)
ax.set_ylabel('Critical Ratio (CR)', fontsize=11)
ax.set_title('Critical Ratio vs Stockout Cost', fontweight='bold')
ax.legend(fontsize=9)

# 2. Avg Order vs p
ax = axes[0, 1]
ax.plot(df_sens['p'], df_sens['Avg Order'], 's-', color='#e67e22', lw=2, markersize=6)
ax.axhline(np.mean(D), color='red', linestyle='--', lw=1, label=f'Mean demand ({np.mean(D):.2f})')
ax.set_xlabel('Stockout Cost (p)', fontsize=11)
ax.set_ylabel('Average Order Quantity', fontsize=11)
ax.set_title('Order Quantity vs Stockout Cost', fontweight='bold')
ax.legend(fontsize=9)

# 3. Fill Rate and Waste% vs p
ax = axes[1, 0]
ax.plot(df_sens['p'], df_sens['Fill Rate'], 'o-', color='#27ae60', lw=2, markersize=6, label='Fill Rate')
ax2 = ax.twinx()
ax2.plot(df_sens['p'], df_sens['Waste%'], 's-', color='#e74c3c', lw=2, markersize=6, label='Waste%')
ax.set_xlabel('Stockout Cost (p)', fontsize=11)
ax.set_ylabel('Fill Rate', fontsize=11, color='#27ae60')
ax2.set_ylabel('Waste %', fontsize=11, color='#e74c3c')
ax.set_title('Fill Rate and Waste vs Stockout Cost', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='center right')

# 4. Profit vs p
ax = axes[1, 1]
ax.plot(df_sens['p'], df_sens['Avg Profit'], 'D-', color='#8e44ad', lw=2, markersize=6)
ax.set_xlabel('Stockout Cost (p)', fontsize=11)
ax.set_ylabel('Average Profit', fontsize=11)
ax.set_title('Profit vs Stockout Cost', fontweight='bold')
ax.axhline(0, color='gray', linestyle=':', lw=1)

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb05_sensitivity_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nKey insights from sensitivity analysis:")
print("  1. Higher stockout cost --> higher CR --> order more units")
print("  2. Fill rate increases with stockout cost (we order more to avoid penalties)")
print("  3. But waste also increases (more excess inventory spoils)")
print("  4. Profit decreases as p increases because penalty costs dominate")
print("  5. This is the FUNDAMENTAL TRADE-OFF: service level vs waste")

---
## Section 8: Key Takeaways

### 1. The Normal Assumption is Convenient but Often Wrong

The classical newsvendor formula $Q^* = \mu + \Phi^{-1}(CR) \cdot \sigma$ is elegant and easy to compute. However, real retail demand frequently violates the Gaussian assumption:
- Many products have zero-demand days (zero inflation)
- Demand is often right-skewed (occasional spikes)
- The Normal PDF assigns positive probability to negative demand

When the distribution assumption is wrong, the resulting order quantities are suboptimal.

### 2. Empirical and Quantile-Based Methods Often Outperform Parametric Approaches

Methods that make fewer assumptions tend to be more robust:
- **Empirical Newsvendor**: Uses actual residuals, no distribution assumption
- **Quantile regression**: Directly predicts the quantiles needed for ordering decisions
- **KDE**: Fully non-parametric density estimation

These methods adapt automatically to the true demand shape, whether it is symmetric, skewed, or zero-inflated.

### 3. The Critical Ratio Drives the Order-Up-To Decision

Regardless of the distributional assumption, the core insight is the same:

$$CR = \frac{\text{underage cost}}{\text{underage cost} + \text{overage cost}}$$

This single number determines where in the demand distribution we should set our order quantity. All policies are ultimately trying to find the $CR$-quantile of the demand distribution -- they just estimate that distribution differently.

### 4. Sensitivity Analysis is Essential

Cost parameters matter as much as forecast accuracy:
- A 2x increase in stockout cost can shift the optimal order by more than a 20% improvement in MAE
- The "right" policy depends on business-specific trade-offs (service level vs waste)
- Always test how your ordering decision changes under different cost assumptions

### Practical Recommendations

| Situation | Recommended Approach |
|-----------|---------------------|
| Lots of historical data | Empirical Newsvendor or KDE |
| Good quantile model | Quantile-Direct Interpolation |
| Many zero-demand products | Zero-Inflated Gamma |
| Simple and fast | Normal Newsvendor with blended sigma |
| Unknown cost parameters | Run sensitivity analysis first |

---
## Section 8: ABC-XYZ Inventory Segmentation

Not all products should be managed the same way. **ABC-XYZ segmentation** classifies products into a 9-cell matrix:

| | **X** (CV ≤ 0.5) | **Y** (0.5 < CV ≤ 1.0) | **Z** (CV > 1.0) |
|---|---|---|---|
| **A** (top 80% volume) | ML + tight SS | ML + moderate SS | ML + generous SS |
| **B** (next 15%) | Simple rules OK | ML recommended | ML + buffer |
| **C** (bottom 5%) | Auto-reorder | Periodic review | Order-on-demand |

- **ABC** = demand volume contribution (Pareto principle: A-class items drive 80% of demand)
- **XYZ** = demand predictability via coefficient of variation (X = smooth, Z = erratic)

This tells planners **where to focus ML effort** and **which items can use simpler rules**.

In [ ]:
# Final summary table
print("\n" + "=" * 100)
print("FINAL SUMMARY: ALL POLICIES RANKED BY PROFIT")
print("=" * 100)
summary = df_all[['Policy', 'Type', 'SL(Type1)', 'Fill Rate', 'Avg Profit', 'SO Rate', 'Waste%']]
summary = summary.sort_values('Avg Profit', ascending=False).reset_index(drop=True)
print(summary.to_string(index=True))

print("\n--- Best Policies ---")
best_profit = summary.iloc[0]
print(f"  Best by profit: {best_profit['Policy']}")
print(f"    Profit={best_profit['Avg Profit']:.4f}, FR={best_profit['Fill Rate']:.4f}, Waste={best_profit['Waste%']:.1f}%")

high_fr = summary[summary['Fill Rate'] >= 0.95]
if len(high_fr) > 0:
    best_hf = high_fr.iloc[0]
    print(f"\n  Best by profit with FR >= 95%: {best_hf['Policy']}")
    print(f"    Profit={best_hf['Avg Profit']:.4f}, FR={best_hf['Fill Rate']:.4f}, Waste={best_hf['Waste%']:.1f}%")

print("\nNotebook complete.")
summary.to_csv(os.path.join(NB_OUT, 'nb05_final_policy_summary.csv'), index=False)
print(f'Saved nb05_final_policy_summary.csv to {NB_OUT}')

In [ ]:
# ---- ABC-XYZ Segmentation ----

# Per-SP statistics from training data
sp_stats = train.groupby('sp').agg(
    total_demand=('sale_amount', 'sum'),
    mean_demand=('sale_amount', 'mean'),
    std_demand=('sale_amount', 'std'),
    n_days=('sale_amount', 'count'),
).reset_index()

sp_stats['std_demand'] = sp_stats['std_demand'].fillna(0)
sp_stats['cv'] = sp_stats['std_demand'] / sp_stats['mean_demand'].clip(lower=0.001)

# ABC classification: by cumulative demand share
sp_stats = sp_stats.sort_values('total_demand', ascending=False).reset_index(drop=True)
sp_stats['cum_share'] = sp_stats['total_demand'].cumsum() / sp_stats['total_demand'].sum()
sp_stats['ABC'] = np.where(sp_stats['cum_share'] <= 0.80, 'A',
                 np.where(sp_stats['cum_share'] <= 0.95, 'B', 'C'))

# XYZ classification: by coefficient of variation
sp_stats['XYZ'] = np.where(sp_stats['cv'] <= 0.5, 'X',
                 np.where(sp_stats['cv'] <= 1.0, 'Y', 'Z'))

sp_stats['Segment'] = sp_stats['ABC'] + sp_stats['XYZ']

# Print distribution
print("ABC Distribution:")
print(sp_stats['ABC'].value_counts().sort_index())
print("\nXYZ Distribution:")
print(sp_stats['XYZ'].value_counts().sort_index())
print("\nSegment Distribution:")
print(sp_stats['Segment'].value_counts().sort_index())
sp_stats.to_csv(os.path.join(NB_OUT, 'nb05_abc_xyz_segmentation.csv'), index=False)
print(f'Saved nb05_abc_xyz_segmentation.csv to {NB_OUT}')

In [ ]:
# ---- ABC-XYZ 9-Cell Heatmap ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ABC-XYZ Inventory Segmentation', fontsize=14, fontweight='bold')

# 1. Heatmap
ax = axes[0]
matrix = np.zeros((3, 3))
abc_labels = ['A', 'B', 'C']
xyz_labels = ['X', 'Y', 'Z']
for i, abc in enumerate(abc_labels):
    for j, xyz in enumerate(xyz_labels):
        matrix[i, j] = (sp_stats['Segment'] == abc + xyz).sum()

im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(3)); ax.set_xticklabels(['X\n(Smooth)', 'Y\n(Variable)', 'Z\n(Erratic)'])
ax.set_yticks(range(3)); ax.set_yticklabels(['A\n(Top 80%)', 'B\n(Next 15%)', 'C\n(Bottom 5%)'])
ax.set_title('ABC-XYZ Matrix (item count)')
for i in range(3):
    for j in range(3):
        v = int(matrix[i, j])
        pct = v / len(sp_stats) * 100
        ax.text(j, i, f'{v}\n({pct:.0f}%)', ha='center', va='center', fontsize=10, fontweight='bold',
                color='white' if matrix[i, j] > matrix.max() * 0.6 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)

# 2. Demand by ABC
ax = axes[1]
for abc, color in zip(['A', 'B', 'C'], ['#c00000', '#ed7d31', '#002060']):
    subset = sp_stats[sp_stats['ABC'] == abc]['mean_demand']
    ax.hist(subset, bins=50, alpha=0.6, label=f'{abc} (n={len(subset)})', color=color, edgecolor='none')
ax.set_xlabel('Mean Daily Demand'); ax.set_ylabel('Count')
ax.set_title('Demand Distribution by ABC Class')
ax.set_xlim(0, sp_stats['mean_demand'].quantile(0.95)); ax.legend()

# 3. CV by XYZ
ax = axes[2]
for xyz, color in zip(['X', 'Y', 'Z'], ['#006b3f', '#ed7d31', '#c00000']):
    subset = sp_stats[sp_stats['XYZ'] == xyz]['cv']
    ax.hist(subset.clip(upper=5), bins=50, alpha=0.6, label=f'{xyz} (n={len(subset)})', color=color, edgecolor='none')
ax.set_xlabel('Coefficient of Variation'); ax.set_ylabel('Count')
ax.set_title('CV Distribution by XYZ Class')
ax.axvline(0.5, color='gray', ls='--', lw=1); ax.axvline(1.0, color='gray', ls='--', lw=1)
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb05_abc_xyz_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\nKey insight: Most items are CZ (low-volume, erratic) — common in retail.")
print("ML forecasting effort should focus on AX/AY/AZ and BX/BY segments.")

---
## Section 9: Before vs After — The Value of ML + Optimization

The most important question for any stakeholder: **"Does this actually work better than what we already do?"**

We compare three approaches to answer this definitively:

1. **Static MinMax (No ML)**: Order = historical mean + 1σ. This simulates traditional fixed reorder points — the "status quo."
2. **ML Forecast Only (No Optimization)**: Use the LightGBM ensemble prediction directly. Shows the value of better predictions alone.
3. **ML + Optimization**: Apply the best inventory policy on top of ML. Shows the full end-to-end value.

The gap between ① and ② shows **the value of ML forecasting**.  
The gap between ② and ③ shows **the value of inventory optimization**.  
The gap between ① and ③ shows **the total value of the full pipeline**.

In [ ]:
# ---- Static Baseline: No ML ----
# Simulate traditional Min/Max: order historical mean + 1 standard deviation
sp_hist_mean = train.groupby('sp')['sale_amount'].mean().to_dict()
sp_hist_std = train.groupby('sp')['sale_amount'].std().fillna(0).to_dict()

Q_static = np.array([sp_hist_mean.get(k, 0) + 1.0 * sp_hist_std.get(k, 0) for k in sp_eval])
Q_static = np.maximum(Q_static, 0)
static_result = eval_policy(Q_static, D, cfg, "Static MinMax (hist μ+σ)")

# Naive ML: just use the forecast directly
naive_result = df_all[df_all['Policy'] == 'Naive (Q=forecast)'].iloc[0].to_dict()

# Best optimized: highest profit among all ML + optimization policies
# (excluding Naive/Q50 Direct which are just raw forecasts, not "optimized")
optimization_policies = df_all[
    ~df_all['Policy'].isin(['Naive (Q=forecast)', 'Q50 Direct', 'Q90 Direct'])
].copy()
best_result = optimization_policies.loc[optimization_policies['Avg Profit'].idxmax()].to_dict()

# ---- Before vs After Comparison ----
comparison = pd.DataFrame([
    {'Approach': '① Static MinMax (No ML)', **static_result},
    {'Approach': '② ML Forecast Only', **naive_result},
    {'Approach': '③ ML + Optimization', **best_result},
])

print("=" * 80)
print("BEFORE vs AFTER: THE VALUE OF ML + OPTIMIZATION")
print("=" * 80)
cols = ['Approach', 'Avg Profit', 'Fill Rate', 'Waste%', 'SO Rate', 'SL(Type1)']
print(comparison[cols].to_string(index=False))

# Compute improvements
ml_gain = (naive_result['Avg Profit'] - static_result['Avg Profit']) / abs(static_result['Avg Profit']) * 100
opt_gain = (best_result['Avg Profit'] - naive_result['Avg Profit']) / abs(naive_result['Avg Profit']) * 100
total_gain = (best_result['Avg Profit'] - static_result['Avg Profit']) / abs(static_result['Avg Profit']) * 100

print(f"\nBest optimization policy:  {best_result['Policy']}")
print(f"Value of ML forecasting:   {ml_gain:>+.1f}% profit over static rules")
print(f"Value of optimization:     {opt_gain:>+.1f}% profit over naive ML")
print(f"Total pipeline value:      {total_gain:>+.1f}% profit improvement end-to-end")

# Show that optimization adds value beyond just profit
fr_gain_opt = (best_result['Fill Rate'] - naive_result['Fill Rate']) * 100
sl_gain_opt = (best_result['SL(Type1)'] - naive_result['SL(Type1)']) * 100
print(f"\n--- Optimization also improves service metrics ---")
print(f"Fill Rate improvement:     {fr_gain_opt:>+.1f} pp ({naive_result['Fill Rate']:.1%} → {best_result['Fill Rate']:.1%})")
print(f"Service Level improvement: {sl_gain_opt:>+.1f} pp ({naive_result['SL(Type1)']:.1%} → {best_result['SL(Type1)']:.1%})")

# Save
comparison.to_csv(os.path.join(NB_OUT, 'nb05_baseline_comparison.csv'), index=False)
print(f'\nSaved nb05_baseline_comparison.csv to {NB_OUT}')


In [ ]:
# ---- Visual: Before vs After Bar Charts (4 metrics) ----
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Before vs After: The Value of ML + Optimization', fontsize=14, fontweight='bold')

labels = ['Static\nMinMax\n(No ML)', 'ML Forecast\nOnly', 'ML +\nOptimization']
colors = ['#666666', '#002060', '#006b3f']

# Profit
ax = axes[0]
profits = [static_result['Avg Profit'], naive_result['Avg Profit'], best_result['Avg Profit']]
bars = ax.bar(labels, profits, color=colors, edgecolor='black', lw=0.5)
ax.set_ylabel('Avg Profit per Unit'); ax.set_title('Profit', fontweight='bold')
for b, v in zip(bars, profits):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.axhline(0, color='red', ls=':', lw=1)

# Fill Rate
ax = axes[1]
frs = [static_result['Fill Rate'], naive_result['Fill Rate'], best_result['Fill Rate']]
bars = ax.bar(labels, [f*100 for f in frs], color=colors, edgecolor='black', lw=0.5)
ax.set_ylabel('Fill Rate (%)'); ax.set_title('Fill Rate', fontweight='bold'); ax.set_ylim(70, 100)
for b, v in zip(bars, frs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')

# Service Level
ax = axes[2]
sls = [static_result['SL(Type1)'], naive_result['SL(Type1)'], best_result['SL(Type1)']]
bars = ax.bar(labels, [s*100 for s in sls], color=colors, edgecolor='black', lw=0.5)
ax.set_ylabel('Service Level (%)'); ax.set_title('Service Level (Type 1)', fontweight='bold'); ax.set_ylim(40, 100)
for b, v in zip(bars, sls):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')

# Waste
ax = axes[3]
wastes = [static_result['Waste%'], naive_result['Waste%'], best_result['Waste%']]
bars = ax.bar(labels, wastes, color=colors, edgecolor='black', lw=0.5)
ax.set_ylabel('Waste (%)'); ax.set_title('Waste', fontweight='bold')
for b, v in zip(bars, wastes):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb05_baseline_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

